# **Etsy Challenge**

---
# Special Colab Setup
### Follow the steps below to setup this colab notebook
---
#### Step 1:
- ???

---
# Step 1: Setup
Setting up imports, downloads, etc.

## 1. Imports

In [9]:
# Takes long to install, best to install in the terminal

# !pip install cleanvision fiftyone
# !pip uninstall onnxruntime onnxruntime-gpu
# !pip install onnxruntime-gpu
# !pip install rembg[gpu]

In [5]:
# General
import setuptools
from tqdm import tqdm
import time
from IPython.display import clear_output
import hashlib


# Data
import pandas as pd  # For Data Exploration and Manipulation
import numpy as np
import kagglehub  # Dataset retrieval


# AI
from fastcore.all import *
from fastai.vision.all import *  # Image Classification
from fastai.callback.progress import ProgressCallback  # Progress Bar
from fastprogress.fastprogress import ConsoleMasterBar, ConsoleProgressBar
from fastai.vision.all import *
import cv2 # OpenCV
from torchvision.models import resnet50 # Resnet50
import torchvision.transforms as T
from tensorflow import keras
from keras.applications.resnet50 import ResNet50, preprocess_input
import onnxruntime # Add this line to import onnxruntime
from rembg import remove, new_session # Segmentation AutoEncoder
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from skimage.feature import graycomatrix, graycoprops
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn as nn
import torch.optim as optim
import timm
import onnxruntime as rt


# Plots
import seaborn as sns  # Plot styling
from matplotlib import pyplot as plt  # Plotting
import plotly.express as px  # Plotting stuff
from plotly.subplots import make_subplots
import plotly.graph_objects as go


# Bens Libraries
from skimage import io # Image I/O, filters, edge detection, resizing
from skimage.color import rgb2gray # Convert RGB images to grayscale
from skimage.filters import threshold_sauvola, gaussian # Global + adaptive thresholding
from skimage import img_as_ubyte # Convert float images to 8‑bit for saving


# Images
import PIL
from PIL import ImageFile, Image
from skimage.feature import hog # Histogram of Oriented Gradients
from skimage import data, exposure


# Data Manipulation
import os  # For general file stuff
import shutil
import zipfile
from pathlib import Path # Helps with different OS Paths
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN # For handling imbalanced datasets
from collections import Counter
import urllib3


# Analysis
import shap
import shap.maskers
from cleanvision import Imagelab # Helps identify image errors
import fiftyone as fo # Visualize our images
import fiftyone.zoo as foz #


# Settings
ImageFile.LOAD_TRUNCATED_IMAGES = True
master_bar, progress_bar = ConsoleMasterBar, ConsoleProgressBar  # Force console-based progress
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Use GPU

#AI sites
import kagglehub


## 2. Setting paths, Extracting Images + Downloading new datasets...

In [6]:
#########################
# Setting paths
#########################

ROOT_DIR = Path("/home/tbeliere/Projects/DCU/AML_PJ")  # or project root if needed
DATA_DIR = ROOT_DIR / "data2"

BASE_DIR = ROOT_DIR 

print(f"Base directory: {BASE_DIR.resolve()}")

challenge_dir = DATA_DIR / "etsy_challenge"
zip_path = challenge_dir / "genai_image_challenge.zip"
final_test_dir = challenge_dir / "final_test_images"
kaggle_dir = DATA_DIR / "other_datasets" / "kaggle_datasets"
created_dirs = DATA_DIR / "other_datasets" / "created_datasets"


if zip_path.exists():
    print("Path exists!")
else:
    raise Exception(f"Path not found: {zip_path}")


# ETSY paths
zip_path = os.path.join(challenge_dir, "genai_image_challenge.zip")
final_test_dir = (
    f"{challenge_dir}/final_test_images"  # extracted here (fast local SSD, not Drive)
)

# Download location paths
shoes_path = f"{kaggle_dir}/shoes_1"
scraped_path = f"{kaggle_dir}/scraped_2"
ai5_path = f"{kaggle_dir}/ai5_3"




#########################
# Unzip etsy dataset
#########################

# Error handler (for zip path)
if os.path.exists(zip_path):
    print("Path exists!\n")
else:
    raise Exception(f"Path not found.")

# Unzip etsy dataset for final test
if not os.path.exists(final_test_dir):  # if directory doesnt exist...
    print("Extracting images...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(final_test_dir)  # extract to images directory
    print(f"Done — extracted to {final_test_dir}\n")
else:
    print(f"Already extracted: {final_test_dir}\n")





#########################
# Download kaggle / other datasets
#########################

# if shoes path doesn't exist
if not os.path.exists(shoes_path):
    ds = kagglehub.dataset_download(
        "sunnykakar/shoes-dataset-real-and-ai-generated-images"
    )  # Dataset 1: AI Shoes

    # Copy dataset to path
    shutil.copytree(ds, shoes_path)

# Else get data from zip
else:
    print("Shoes path exists!\n")


# if scraped path doesn't exist
if not os.path.exists(scraped_path):
    ds = kagglehub.dataset_download(
        "cashbowman/ai-generated-images-vs-real-images"
    )  # Dataset 2: AI Scraped

    # Copy dataset to path
    shutil.copytree(ds, scraped_path)

# Else get data from zip
else:
    print("scraped path exists!\n")


# if shoes path doesn't exist
if not os.path.exists(ai5_path):
    ds = kagglehub.dataset_download(
        "rhythmghai/ai-vs-real-images-dataset"
    )  # Dataset 3: AI 5 categories

    # Copy dataset to path
    shutil.copytree(ds, ai5_path)

# Else get data from zip
else:
    print("AI 5 path exists!\n")





#########################
# Where are the datasets?
#########################

# Where are the datasets?
print(f"Path to the Shoes dataset: {shoes_path}")
print(f"Path to the Ai vs Web-scraped dataset: {scraped_path}")
print(f"Path to the Ai (5 categories) dataset: {ai5_path}\n")

Base directory: /home/tbeliere/Projects/DCU/AML_PJ
Path exists!
Path exists!

Already extracted: /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge/final_test_images

Path to the Shoes dataset: /home/tbeliere/Projects/DCU/AML_PJ/data2/other_datasets/kaggle_datasets/shoes_1
Path to the Ai vs Web-scraped dataset: /home/tbeliere/Projects/DCU/AML_PJ/data2/other_datasets/kaggle_datasets/scraped_2
Path to the Ai (5 categories) dataset: /home/tbeliere/Projects/DCU/AML_PJ/data2/other_datasets/kaggle_datasets/ai5_3



## 3. Unzipping and Cleaning datasets
by unzipping them and cleaning up their folder structure

In [1]:
from pathlib import Path
import os
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
print("Initial PROJECT_ROOT:", PROJECT_ROOT)
# If notebook is inside notebooks/, move one level up automatically
if (PROJECT_ROOT / "data2").exists():
    pass
elif (PROJECT_ROOT.parent / "data2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise FileNotFoundError("Could not locate project root containing data2/")

CHALLENGE_DIR = PROJECT_ROOT / "data2" / "etsy_challenge"
TRAIN_CSV = CHALLENGE_DIR / "train.csv"
TEST_CSV = CHALLENGE_DIR / "test.csv"
TRAIN_IMG_DIR = CHALLENGE_DIR / "train_images"
TEST_IMG_DIR = CHALLENGE_DIR / "final_test_images"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CHALLENGE_DIR:", CHALLENGE_DIR)

Initial PROJECT_ROOT: /home/tbeliere/Projects/DCU/AML_PJ/src/daniels_files/combined_notebook


FileNotFoundError: Could not locate project root containing data2/

In [2]:
#########################
# Setting paths
#########################
from pathlib import Path
import zipfile
import shutil
import os

PROJECT_ROOT = Path("/home/tbeliere/Projects/DCU/AML_PJ")
print("PROJECT_ROOT:", PROJECT_ROOT)

BASE_DIR = PROJECT_ROOT

# Main directories
challenge_dir = BASE_DIR / "data2" / "etsy_challenge"
zip_path = challenge_dir / "genai_image_challenge.zip"
final_test_dir = challenge_dir / "final_test_images"

other_datasets_dir = BASE_DIR / "other_datasets"
kaggle_dir = other_datasets_dir / "kaggle_datasets"
created_dir = other_datasets_dir / "created_datasets"

# Download target directories
shoes_path = kaggle_dir / "shoes_1"
scraped_path = kaggle_dir / "scraped_2"
ai5_path = kaggle_dir / "ai5_3"

# Create parent directories if needed
challenge_dir.mkdir(parents=True, exist_ok=True)
kaggle_dir.mkdir(parents=True, exist_ok=True)
created_dir.mkdir(parents=True, exist_ok=True)

# Sanity prints
print(f"Challenge directory: {challenge_dir}")
print(f"Zip path: {zip_path}")
print(f"Final test directory: {final_test_dir}")
print(f"Kaggle datasets directory: {kaggle_dir}")
print(f"Created datasets directory: {created_dir}")
print()

# Check Etsy zip exists
if zip_path.exists():
    print("Etsy zip path exists!\n")
else:
    raise FileNotFoundError(f"Etsy zip not found: {zip_path}")


#########################
# Unzip Etsy dataset
#########################

if not final_test_dir.exists():
    print("Extracting Etsy images...")
    final_test_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(final_test_dir)

    print(f"Done — extracted to {final_test_dir}\n")
else:
    print(f"Already extracted: {final_test_dir}\n")


#########################
# Download Kaggle / other datasets
#########################

# Dataset 1: Shoes
if not shoes_path.exists():
    print("Downloading Shoes dataset...")
    ds = kagglehub.dataset_download("sunnykakar/shoes-dataset-real-and-ai-generated-images")
    shutil.copytree(ds, shoes_path)
    print(f"Saved Shoes dataset to: {shoes_path}\n")
else:
    print(f"Shoes path exists: {shoes_path}\n")

# Dataset 2: Web-scraped
if not scraped_path.exists():
    print("Downloading AI vs Web-scraped dataset...")
    ds = kagglehub.dataset_download("cashbowman/ai-generated-images-vs-real-images")
    shutil.copytree(ds, scraped_path)
    print(f"Saved Scraped dataset to: {scraped_path}\n")
else:
    print(f"Scraped path exists: {scraped_path}\n")

# Dataset 3: AI 5 categories
if not ai5_path.exists():
    print("Downloading AI 5 categories dataset...")
    ds = kagglehub.dataset_download("rhythmghai/ai-vs-real-images-dataset")
    shutil.copytree(ds, ai5_path)
    print(f"Saved AI5 dataset to: {ai5_path}\n")
else:
    print(f"AI5 path exists: {ai5_path}\n")


#########################
# Where are the datasets?
#########################

print("Final dataset locations:")
print(f"Path to the Etsy extracted images: {final_test_dir}")
print(f"Path to the Shoes dataset: {shoes_path}")
print(f"Path to the AI vs Web-scraped dataset: {scraped_path}")
print(f"Path to the AI (5 categories) dataset: {ai5_path}")
print(f"Path to created datasets directory: {created_dir}")

PROJECT_ROOT: /home/tbeliere/Projects/DCU/AML_PJ
Challenge directory: /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge
Zip path: /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge/genai_image_challenge.zip
Final test directory: /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge/final_test_images
Kaggle datasets directory: /home/tbeliere/Projects/DCU/AML_PJ/other_datasets/kaggle_datasets
Created datasets directory: /home/tbeliere/Projects/DCU/AML_PJ/other_datasets/created_datasets

Etsy zip path exists!

Already extracted: /home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_challenge/final_test_images



NameError: name 'kagglehub' is not defined

## 4. Joining our datasets together to main_df
This is to get a better sense of image sizes, imbalances, etc.

In [ ]:
#########################
# Folder check + initializing / declaring variables
#########################

# What folders do we have?
print(f"Shoes folders:\n\t{os.listdir(shoes_path)}")
print(f"Web scraped folders:\n\t{os.listdir(scraped_path)}")
print(f"5 categories folders:\n\t{os.listdir(ai5_path)}\n")


# This will be used for creating the dataframes
image_paths = []
labels = []





#########################
# Make dataframes
#########################

# Make df from folders + images
for folder in os.listdir(shoes_path): # For folders in garbage dataset

    # For every image
    for image in os.listdir(shoes_path + "/" + folder):

        # Get image paths
        image_paths.append(os.path.join(shoes_path + "/" + folder, image))
        
        # Add it to a list
        labels.append(folder)


# Make dataframe from paths + labels        
shoes_df = pd.DataFrame({"path": image_paths, "label": labels})
image_paths, labels = [], [] # clear our arrays



# Make df from folders + images
for folder in os.listdir(scraped_path): # For folders in TrashNet dataset

    # For every image
    for image in os.listdir(scraped_path + "/" + folder):

        # Get image paths
        image_paths.append(os.path.join(scraped_path + "/" + folder, image))

        # Add it to a list
        labels.append(folder)


# Make dataframe from paths + labels  
scraped_df = pd.DataFrame({"path": image_paths, "label": labels})
image_paths, labels = [], [] # Clear our arrays



# Make df from folders + images
for folder in os.listdir(ai5_path): # For folders in RealWaste dataset

    # For every image
    for image in os.listdir(ai5_path + "/" + folder):

        # Get image paths
        image_paths.append(os.path.join(ai5_path + "/" + folder, image))

        # Add it to a list
        labels.append(folder)


# Make dataframe from paths + labels
ai5_df = pd.DataFrame({"path": image_paths, "label": labels})
image_paths, labels = [], [] # Clear our arrays





#########################
# Normalize labels, check labels + join dataframes to main_df
#########################

# Normalize by making lowercase
def normalize_label(label):
    return label.strip().lower()


# Convert all labels to lowercase
for df in [shoes_df, scraped_df, ai5_df]:
    df['label'] = df['label'].apply(normalize_label)


# Check labels
print(f"\nNormalized Garbage DF Columns: \n\t{shoes_df['label'].unique()}")
print(f"Normalized Trash DF Columns: \n\t{scraped_df['label'].unique()}")
print(f"Normalized Real Waste DF Columns: \n\t{ai5_df['label'].unique()}")


# Join our datasets
frames = [shoes_df, scraped_df, ai5_df]
main_df = pd.concat(frames, ignore_index=True)


# Check main_df labels
print(f"Main DF Columns: \n\t{main_df['label'].unique()}\n")
print(f"Main DF Info: \n\t{main_df.info()}\n")

## 5: Data Splitting Preparation

In [ ]:
#########################
# Getters
#########################

# Get image
def get_x(r): return r['path']

# Get waste type
def get_y(r): return r['label']





#########################
# Split Train DF
#########################

# Split new_main_df (70% Train)
train_df, temp_df = train_test_split(
    main_df,
    test_size=0.30,
    random_state=42,
    stratify=main_df['label']
)

train_df.head()

## 6. Creating "data/MainData" for joined Data Folder
So we have our images in one place

In [ ]:
#########################
# "Create MainData folder" function
#########################

# Create folder from image paths
def create_folder_from_df(df: pd.DataFrame.T, file_path: string):

    # Make a folder with all images
    if (not Path(file_path).exists()):
        counts = {}
        for path, label in df.values: # For every path and class
            
            # If no label
            if label not in counts:
                counts[label] = 0 # Add one
            else:
                counts[label] += 1 # Increment value

            # if path doesn't exist
            if not os.path.isdir(f"{file_path}/{label}"):
                os.makedirs(f"{file_path}/{label}") # Make the directory

            # Copy images over + rename
            _, extension = os.path.splitext(path) # Get the name and extension...
            shutil.copyfile(path, f"{file_path}/{label}/{label}_{counts[label]}{extension}")

    # Else, print status        
    else:
        print("MainData already exists")





#########################
# Convert organized folders to Labels
#########################

# What classes do we want?
classes = {
    # Organics
    "ai": "1",

    # Textiles
    "real": "0",
}


# Replace labels with appropriate classes
train_df['label'] = train_df['label'].replace(classes)
temp_df['label'] = temp_df['label'].replace(classes)

# Remain unchanged for better generalization
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.3333,
    random_state=42,
    stratify=temp_df['label']
)





#########################
# Create MainData folder + New Main DF and Check
#########################

# New df from folders
create_folder_from_df(train_df, f"{main_data_location}") # Create new folders
new_main_df = pd.DataFrame(
    [(f"{main_data_location}/{label}/{f}", label) for label in os.listdir(f"{main_data_location}") for f in os.listdir(f"{main_data_location}/{label}")], 
    columns=["path","label"])


# Quick peak into df
new_main_df.head()

## 7. Exploring Random Image Samples

In [ ]:
# What work needs to be done
imagelab = Imagelab(data_path=main_data_location) # Specify path to folder 
imagelab.find_issues() 
imagelab.report() # Produce a report of problems

In [ ]:
# Get flagged images with their duplicate scores
exact_duplicates = imagelab.issues[imagelab.issues['is_exact_duplicates_issue'] == True][
    ['is_exact_duplicates_issue', 'exact_duplicates_score']
]
print(f"Total: {len(exact_duplicates)}")
print(exact_duplicates)

In [ ]:
# Get exact duplicate scores for all flagged images
exact_duplicates = imagelab.issues[imagelab.issues['is_exact_duplicates_issue'] == True]['exact_duplicates_score']

print(f"Highest score: {exact_duplicates.max()}")
print(f"Lowest score:  {exact_duplicates.min()}")
print(f"Image with highest score: {exact_duplicates.idxmax()}")
print(f"Image with lowest score:  {exact_duplicates.idxmin()}")

In [ ]:
#########################
# Get scores
#########################

# Get images (highest / lowest duplicate scores)
exact_duplicates = imagelab.issues[imagelab.issues['is_exact_duplicates_issue'] == True]['exact_duplicates_score']
highest_img = exact_duplicates.idxmax()
lowest_img = exact_duplicates.idxmin()





#########################
# Find which sets own the images
#########################

# Find sets these images belong to
duplicate_sets = imagelab.info['exact_duplicates']['sets']
highest_set = None
lowest_set = None


for s in duplicate_sets:
    if highest_img in s:
        highest_set = s
    if lowest_img in s:
        lowest_set = s





#########################
# Create image set display function
#########################

# Shows the highest / Lowest image sets
def show_image_set(image_set, title):
    fig, axes = plt.subplots(1, len(image_set), figsize=(5 * len(image_set), 5))
    if len(image_set) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, image_set):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(img_path.split('/')[-1], fontsize=8)
        ax.axis('off')
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()





#########################
# Check score + display
#########################

# Check highest score + Call show function
print(f"Highest score image: {highest_img} (score: {exact_duplicates.max()})")
show_image_set(highest_set, f"Set containing highest scoring image")

# Check lowest score + Call show function 
print(f"Lowest score image: {lowest_img} (score: {exact_duplicates.min()})")
show_image_set(lowest_set, f"Set containing lowest scoring image")

In [ ]:
# Check all sets
duplicate_sets = imagelab.info['exact_duplicates']['sets']
for i, s in enumerate(duplicate_sets):
    print(f"Set {i}: {s}")

### Remove duplicates and keeping only one image per image set

In [ ]:
#########################
# Get images + normalize path 
#########################

# Get all images to remove (keeping first in each set)
duplicate_sets = imagelab.info['exact_duplicates']['sets']


to_remove = []
for s in duplicate_sets:
    to_remove.extend(s[1:])


# Normalize to_remove paths to match new_main_df format
to_remove_normalized = [p.replace('/content/', '') for p in to_remove]





#########################
# Get original size + delete duplicates
#########################

# Capture original size before filtering
original_size = len(new_main_df)


# Filter them out and rebind new_main_df
new_main_df = new_main_df[~new_main_df['path'].isin(to_remove_normalized)].reset_index(drop=True)





#########################
# Check / compare New Main DF
#########################

print(f"Original dataframe size: {original_size}")
print(f"Duplicates removed: {len(to_remove)}")
print(f"New dataframe size: {len(new_main_df)}")

In [ ]:
#########################
# Add hash to images
#########################

# Returns hash for an image 
def hash_image(path):
    with Image.open(path) as img:
        return hashlib.md5(img.tobytes()).hexdigest()


# Use function on images
new_main_df['hash'] = new_main_df['path'].apply(hash_image) 





#########################
# Check for duplicates
#########################

# Duplicate images variable
duplicate_images = new_main_df[new_main_df.duplicated(subset=['hash'], keep=False)]

# Check for duplicate images
print(f"Duplicate images found: {len(duplicate_images)}")
duplicate_images

In [ ]:
# Drop hash column
try:
    new_main_df = new_main_df.drop('hash', axis=1)
except:
    print('Hash column has been dropped')
new_main_df.head()

---
# Step 2: General Exploration

Exploring our data

## 1. General Overview

### Describe

In [ ]:
# General Overview
new_main_df.describe()

### Info

In [ ]:
# Column Datatypes?
new_main_df.info()

### Checking Existing Labels

In [ ]:
# Labels we have so far?
print(f"found {new_main_df['label'].nunique()} different image labels:")
print(f"\n\n\t{new_main_df['label'].unique()}")

## 2. Class Imbalances

### Overview

In [ ]:
# Overview of class imbalances
print(f"Number of images? -> {len(new_main_df)}")
print(f"Number of trash types? -> {len(new_main_df['label'].value_counts())}")
print(f'\n\nNumber of images for each type?')
print(f'------\n{new_main_df["label"].value_counts()}')

### Histogram

In [ ]:
# Histogram
fig, ax = plt.subplots(figsize=(15, 9))
sns.countplot(data=new_main_df, x='label', order=new_main_df['label'].value_counts().index)
plt.title('Number of images for each Trash Type')
plt.xlabel('\nTrash type')
plt.ylabel('Number of Images')

### Pie Chart

In [ ]:
# Get labels + their counts
type_vc = new_main_df['label'].value_counts(sort=False)
total = type_vc.sum()


# Pie chart
fig = px.pie(type_vc, 
    values=type_vc.values, 
    names=type_vc.index, 
    hole=0.500
    )
fig.update_layout(
    title='Data Distribution of Main Dataset', 
    font_size=10, 
    title_x=0.40,
    annotations=[dict(
        text='Main Dataset', 
        font_size=12, 
        showarrow=False, 
        height=5000, 
        width=5000)]
    )
fig.update_traces(
    textfont_size=8, 
    textinfo='label+percent+text',
    text=[f"1:{round(total / v)}" for v in type_vc.values] # Get ratio for every value
    ) 
fig.show()

## 3. Transforms

In [ ]:
# Set augment variable for later training
aug = aug_transforms(
    do_flip=True,       
    max_rotate=15.0,    
    max_zoom=1.0,       
    max_warp=0.0,       
    max_lighting=0.2    # This is fastai's default
)

## 4. Dealing with Imbalances

### Random Over Sampling
#### Advantages
- Very simple and fast
- Best for small datasets
#### Disadvantages
- High Overfitting risk
- Duplicates randomly chosen images

In [ ]:
#########################
# Set sampler + resample data + new DF
#########################

# Setup the sampler
ros = RandomOverSampler(random_state=42, sampling_strategy="all")


# Resample the data
X_resampled, y_resampled = ros.fit_resample(new_main_df[['path']], new_main_df['label'])


# Create new balanced dataframe
ros_balanced_df = pd.DataFrame({
    'path': X_resampled['path'],
    'label': y_resampled
})





#########################
# Check counts + plot counts
#########################

# increased counts shown
print(f"Number of images? -> {len(ros_balanced_df)}")
print(f"Counts per type:\n{ros_balanced_df['label'].value_counts()}")


# Quick plot to show the difference
fig, axes = plt.subplots(1, 2, figsize=(15, 5)) # plot image + axes


# Rotate labels 45 degrees 
for ax in axes:
    ax.set_xticklabels(new_main_df['label'].unique(), rotation=45, ha='right')
    ax.set_xlabel('Trash Type')
    ax.set_ylabel('Number of Images')


# Original distribution
axes[0].bar(Counter(new_main_df['label']).keys(), Counter(new_main_df['label']).values(), color='b')
axes[0].set_title('Original Distribution')


# After upsampling
axes[1].bar(Counter(y_resampled).keys(), Counter(y_resampled).values(), color='g')
axes[1].set_title('After Upsampling')


plt.tight_layout()
plt.show()

### Weighted Random Sampling
#### Advantages
- Assigns weights to minority classes
- Does not change the size of the dataset
#### Disadvantages
- Severe overfitting
- Majority class is represented less

In [ ]:
#########################
# Counts + calculate probabilities and weights
#########################

# Count occurrences per class
class_counts = new_main_df['label'].value_counts()
total_images = len(new_main_df)

# Calculate natural unweighted probability
orig_prob = (class_counts / total_images) * 100

# Calculate balanced sampler weights
class_weights = 1.0 / class_counts

# Normalize weights to percentages
new_prob = (class_weights / class_weights.sum()) * 100





#########################
# Comparison DF + plot probabilities
#########################

# Create simple comparison dataframe
compare_df = pd.DataFrame({
    'Physical Images': class_counts,
    'Original Chance (%)': orig_prob,
    'Weighted Chance (%)': new_prob
}).round(2)

# Show the final comparison
print(f"Total physical images: {total_images}")
print(compare_df)

# Plot the before/after probabilities
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot original draw chances
axes[0].bar(orig_prob.index, orig_prob.values, color='b')
axes[0].set_title('Original Draw Chance (%)')
axes[0].set_ylabel('Probability (%)')

# Plot balanced draw chances
axes[1].bar(new_prob.index, new_prob.values, color='g')
axes[1].set_title('Weighted Draw Chance (%)')
axes[1].set_ylabel('Probability (%)')

# Rotate labels 45 degrees
for ax in axes:
    ax.tick_params(axis='x', rotation=45)

# Show the final graphs
plt.tight_layout()
plt.show()

In [ ]:
#########################
# Check weights, set variable + create sampler
#########################

print(f"{class_weights}\n")

# map weights to labels
sample_weights = new_main_df['label'].map(class_weights)

# Convert pandas to PyTorch tensor
weights_tensor = torch.tensor(sample_weights.values, dtype=torch.float)

# Create the actual PyTorch sampler
sampler = WeightedRandomSampler(
    weights=weights_tensor, 
    num_samples=len(weights_tensor), 
    replacement=True
)

print(f"Sampler created with {len(weights_tensor)} weights.")

### SMOTE
#### Advantages
- Adds diversity (lowers overfitting risk)
- Best for deep features
#### Disadvantages
- Slower but more complex
- Synthesizes new images (which look like ghost images, or two images joined together)

In [ ]:
smote_folder = f"{created_dir}/SmoteData"
os.makedirs(smote_folder, exist_ok=True)

target_size = (128, 128)
pixel_data = []
labels = []

for idx, row in new_main_df.iterrows():
    img = cv2.imread(str(row['path']))
    if img is not None:
        img = cv2.resize(img, target_size)
        pixel_data.append(img.flatten())
        labels.append(row['label'])

X = np.array(pixel_data)
y = np.array(labels)

smote = SMOTE(random_state=42, sampling_strategy="all")
X_res, y_res = smote.fit_resample(X, y)

new_paths = []
for i, (flat_img, label) in enumerate(zip(X_res, y_res)):
    img = flat_img.reshape(target_size[0], target_size[1], 3).astype(np.uint8)
    os.makedirs(f"{smote_folder}/{label}", exist_ok=True)
    filename = f"synthetic_{label}_{i}.jpg"
    filepath = os.path.join(smote_folder, label, filename)
    cv2.imwrite(filepath, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    new_paths.append(filepath)

smote_balanced_df = pd.DataFrame({'path': new_paths, 'label': y_res})

print(f"Total images: {len(smote_balanced_df)}")
print(smote_balanced_df['label'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
orig_counts = Counter(y)
res_counts = Counter(y_res)

axes[0].bar(orig_counts.keys(), orig_counts.values(), color='b')
axes[0].set_title('Original')
axes[1].bar(res_counts.keys(), res_counts.values(), color='g')
axes[1].set_title('SMOTE Balanced')

for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

sample_img = cv2.imread(new_paths[-1])
plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
plt.title("Example of SMOTE 'Ghost' Image")
plt.axis('off')
plt.show()

### Deep SMOTE
#### Advantages
- Uses an Autoencoder and Decoder
- Overall improvement over Smote
#### Disadvantages

In [ ]:
# Class setup
class DeepSMOTE_Autoencoder(nn.Module):
    def __init__(self, latent_dim=1024):
        super(DeepSMOTE_Autoencoder, self).__init__()
        self.latent_dim = latent_dim
        
        # Encoder (Downsampling)
        self.encoder = nn.Sequential(
            # 224x224 -> 112x112
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            
            # 112x112 -> 56x56
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            
            # 56x56 -> 28x28
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            # 28x28 -> 14x14
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            
            # 14x14 -> 7x7
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            
            nn.Flatten() 
            # Output of flatten -> 512 channels * 7 height * 7 width = 25088
        )
        
        # Squash 25088 features into Latent Vector
        self.fc_encode = nn.Linear(512 * 7 * 7, self.latent_dim) # 1024
        
		# Decoder (Upsampling)
        # Expand  latent vector back to 25088
        self.fc_decode = nn.Linear(self.latent_dim, 512 * 7 * 7)
        
        self.decoder = nn.Sequential(
            # 7x7 -> 14x14
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            
            # 14x14 -> 28x28
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            # 28x28 -> 56x56
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            
            # 56x56 -> 112x112
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            
            # 112x112 -> 224x224
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid() 
        )

    def forward(self, x):
        # Encode
        x = self.encoder(x)
        latent = self.fc_encode(x)
        
        x = self.fc_decode(latent) # Decode
        
        x = x.view(-1, 512, 7, 7) # Reshape flat array to 2D grid 
        reconstructed_image = self.decoder(x)
        
        return reconstructed_image, latent

In [ ]:
# Setup standard image transformations
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
])

# Hold all processed tensor data
all_images = []

print("Starting pre-processing... Please wait.")

# Iterate through entire dataset dataframe
for idx, row in new_main_df.iterrows():
    img_path = row['path']
    
    # Load, convert, and transform image
    try:
        img = Image.open(img_path).convert('RGB')
        img_t = transform(img)
        all_images.append(img_t)
    except Exception as e:
        print(f"Skipping bad image: {img_path}")

# Stack into one massive tensor
print("Stacking tensors into memory...")
stacked_images = torch.stack(all_images)

# Save tensor directly to disk
print("Saving to disk...")
torch.save(stacked_images, 'fast_waste_images.pt')

print("Saved successfully as fast_waste_images.pt!")

In [ ]:
# Define the model save path
model_path = 'deepsmote_autoencoder.pth'

# Initialize model, loss, and optimizer
model = DeepSMOTE_Autoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Load massive tensor into RAM
fast_data = torch.load('fast_waste_images.pt')

# Create ultra-fast custom dataset
class FastImageDataset(Dataset):
	def __init__(self, tensor_data):
		self.images = tensor_data

	def __len__(self):
		return len(self.images)

	def __getitem__(self, idx):
		# Return pre-computed tensor directly
		return self.images[idx]

# Initialize the fast dataset
ae_dataset = FastImageDataset(fast_data)

# Create optimized CPU-free DataLoader
ae_loader = DataLoader(
    ae_dataset, batch_size=256, shuffle=True, num_workers=0, pin_memory=True
)

# Check if model already exists
if os.path.exists(model_path):
    # Load the existing model weights
    model.load_state_dict(torch.load(model_path))
    # Print skip training message
    print("Model found! Skipping training process.")
    
else:

    # Initialize scaler for mixed precision
    scaler = torch.amp.GradScaler(device.type)

    # Number of training epochs needed
    num_epochs = 500

    # Print the starting training message
    print("Starting Autoencoder Training...")

    # Enable cudnn benchmark for speed
    torch.backends.cudnn.benchmark = True

    # Allow manual stopping and saving
    try:
        for epoch in range(num_epochs):
            # Record epoch start time
            start_time = time.time()
            
            # Set model to training mode
            model.train()
            running_loss = 0.0

            for images in ae_loader:
                # Move images to current device
                images = images.to(device)

                # Zero the optimizer gradients
                optimizer.zero_grad()

                with torch.amp.autocast(device.type):
                    # Encode images into latent vectors
                    features = model.encoder(images)
                    latent = model.fc_encode(features)

                    # Decode vectors back into images
                    decoded_features = model.fc_decode(latent).view(-1, 512, 7, 7)
                    reconstructed_images = model.decoder(decoded_features)

                    # Calculate mean squared error loss
                    loss = criterion(reconstructed_images, images)

                # Scale loss and perform backprop
                scaler.scale(loss).backward()

                # Step optimizer and update scaler
                scaler.step(optimizer)
                scaler.update()

                # Add to running epoch loss
                running_loss += loss.item()

            # Calculate elapsed epoch time
            end_time = time.time()
            epoch_duration = end_time - start_time

            # Calculate and print average loss
            avg_loss = running_loss / len(ae_loader)
            
            # Clear notebook cell output
            clear_output(wait=True)
            
            # Print loss and elapsed time
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Time: {epoch_duration:.2f}s")
            print("Click STOP to save model.")
            
            # Convert to float to avoid ValueError
            orig_img = images[0].cpu().detach().float()
            
            # Convert to float to avoid ValueError
            recon_img = reconstructed_images[0].cpu().detach().float()
            
            # Rearrange channels for matplotlib
            orig_img = orig_img.permute(1, 2, 0).numpy()
            recon_img = recon_img.permute(1, 2, 0).numpy()
            
            # Clip values for safe plotting
            orig_img = np.clip(orig_img, 0, 1)
            recon_img = np.clip(recon_img, 0, 1)
            
            # Create a new figure
            fig, axes = plt.subplots(1, 2, figsize=(10, 5))
            
            # Set the epoch title
            fig.suptitle(f"Epoch {epoch+1} Reconstruction", fontsize=14, fontweight="bold")
            
            # Plot the original image
            axes[0].imshow(orig_img)
            axes[0].set_title("Original Image")
            axes[0].axis('off')
            
            # Plot the reconstructed image
            axes[1].imshow(recon_img)
            axes[1].set_title("Reconstructed Image")
            axes[1].axis('off')
            
            # Adjust layout and show2
            plt.tight_layout()
            plt.show()

    # Catch manual stop signal
    except KeyboardInterrupt:
        # Print early stopping message
        print("\nTraining manually stopped by user!")

    # Print final training complete message
    print("Training Complete!")

    # Save trained model weights
    torch.save(model.state_dict(), model_path)

    # Print final save message
    print(f"Model saved to {model_path}")

In [ ]:
# Set model to evaluation
model.eval()

# Get one batch of images
dataiter = iter(ae_loader)
images = next(dataiter)

# Move images to GPU
images = images.to(device)

with torch.no_grad():
    # Encode and decode images
    features = model.encoder(images)
    latent = model.fc_encode(features)
    
    # Reshape and decode back
    decoded = model.fc_decode(latent).view(-1, 512, 7, 7)
    reconstructed = model.decoder(decoded)

# Move tensors to CPU
images = images.cpu()
reconstructed = reconstructed.cpu()

# Plot original and reconstructed
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i in range(5):
    # Display original top row
    ax = axes[0, i]
    img = images[i].permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title("Original")
    ax.axis('off')
    
    # Display reconstructed bottom row
    ax = axes[1, i]
    recon = reconstructed[i].permute(1, 2, 0).numpy()
    recon = np.clip(recon, 0, 1) # Clip valid pixel range
    ax.imshow(recon)
    ax.set_title("Reconstructed")
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
smote_folder = "data/DeepSmoteData"
os.makedirs(smote_folder, exist_ok=True)

target_size = (224, 224)  # Must be 224 for autoencoder
latent_data = []  # Store compressed latent vectors
labels = []

# model = DeepSMOTE_Autoencoder().to(device)


model.eval()  # Set model to evaluation

for idx, row in new_main_df.iterrows():
    img = cv2.imread(str(row["path"]))
    if img is not None:
        img = cv2.resize(img, target_size)

        # Convert BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Encode image to latent
        img_t = T.ToTensor()(img).unsqueeze(0).to(device)

        # Encode image to latent
        img_t = T.ToTensor()(img).unsqueeze(0).to(device)
        with torch.no_grad():
            features = model.encoder(img_t)
            latent = model.fc_encode(features)

        latent_data.append(latent.cpu().numpy().flatten())
        labels.append(row["label"])

X = np.array(latent_data)
y = np.array(labels)

# Apply SMOTE to latent space
smote = SMOTE(random_state=42, sampling_strategy="all")
X_res, y_res = smote.fit_resample(X, y)

new_paths = []

# Decode latent vectors back
for i, (latent_vec, label) in enumerate(zip(X_res, y_res)):
    vec_t = torch.tensor(latent_vec, dtype=torch.float32).unsqueeze(0).to(device)

    # Decode back into image
    with torch.no_grad():
        decoded_features = model.fc_decode(vec_t).view(-1, 512, 7, 7)
        synthetic_img = model.decoder(decoded_features).squeeze(0)

    img = (synthetic_img.cpu().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)

    os.makedirs(f"{smote_folder}/{label}", exist_ok=True)
    filename = f"synthetic_{label}_{i}.jpg"
    filepath = os.path.join(smote_folder, label, filename)
    cv2.imwrite(filepath, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    new_paths.append(filepath)

deep_smote_balanced_df = pd.DataFrame({"path": new_paths, "label": y_res})

print(f"Total images: {len(deep_smote_balanced_df)}")
print(deep_smote_balanced_df["label"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
orig_counts = Counter(y)
res_counts = Counter(y_res)

axes[0].bar(orig_counts.keys(), orig_counts.values(), color="b")
axes[0].set_title("Original")
axes[1].bar(res_counts.keys(), res_counts.values(), color="g")
axes[1].set_title("Deep SMOTE Balanced")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

sample_img = cv2.imread(new_paths[-1])
plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
plt.title("Example of Deep SMOTE 'Ghost' Image")
plt.axis("off")
plt.show()

### ADASYN
#### Advantages
- An extension of SMOTE
- Adds diversity, similar to SMOTE (lowers overfitting risk)
- Focuses on more rare or hard to predict classes
#### Disadvantages
- More complex than SMOTE, but even slower
- May focus more on extreme outliers than SMOTE, which can be bad, as this creates noise

In [ ]:
adasyn_folder = f"{created_dir}/AdasynData"
os.makedirs(adasyn_folder, exist_ok=True)

target_size = (128, 128)
pixel_data = []
labels = []

for idx, row in new_main_df.iterrows():
    img = cv2.imread(str(row['path']))
    if img is not None:
        img = cv2.resize(img, target_size)
        pixel_data.append(img.flatten())
        labels.append(row['label'])

X = np.array(pixel_data)
y = np.array(labels)


# Check if the dataset is balanced
counts = Counter(y)
if len(set(counts.values())) > 1:
    adasyn = ADASYN(random_state=42, sampling_strategy="not majority", n_neighbors=2)
    X_res, y_res = adasyn.fit_resample(X, y)
else:
    print("Dataset is already balanced. Skipping ADASYN.")
    X_res, y_res = X, y


new_paths = []
for i, (flat_img, label) in enumerate(zip(X_res, y_res)):
    img = flat_img.reshape(target_size[0], target_size[1], 3).astype(np.uint8)
    os.makedirs(f"{adasyn_folder}/{label}", exist_ok=True)
    filename = f"adasyn_{label}_{i}.jpg"
    filepath = os.path.join(adasyn_folder, label, filename)
    cv2.imwrite(filepath, img)
    new_paths.append(filepath)

adasyn_balanced_df = pd.DataFrame({'path': new_paths, 'label': y_res})

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
orig_counts = Counter(y)
res_counts = Counter(y_res)

axes[0].bar(orig_counts.keys(), orig_counts.values(), color='royalblue')
axes[0].set_title('Original')
axes[1].bar(res_counts.keys(), res_counts.values(), color='seagreen')
axes[1].set_title('ADASYN Balanced')

for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

sample_img = cv2.imread(new_paths[-1])
plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
plt.title("ADASYN Synthetic Sample")
plt.axis('off')
plt.show()

### DUPLICATION (DEPRECATED)
by duplicating less common classes (Oversampling)

In [ ]:
# max_size = new_main_df['label'].value_counts().max()
# balanced_df = new_main_df.groupby('label').sample(n=max_size, replace=True).reset_index(drop=True)
# # Overview of class imbalances
# print(f"Number of images? -> {len(balanced_df)}")
# print(f"Number of trash types? -> {len(balanced_df['label'].value_counts())}")
# print(f'\n\nNumber of images for each type?')
# print(f'------\n{balanced_df["label"].value_counts()}')

### Check Again (Random Over Sampling)
Use a Histogram and Pie Chart to see our classes 

In [ ]:
# Histogram
fig, ax = plt.subplots(figsize=(15, 9))
sns.countplot(data=ros_balanced_df, x='label', order=ros_balanced_df['label'].value_counts().index)
plt.title('Number of images for each Trash Type')
plt.xlabel('\nTrash type')
plt.ylabel('Number of Images')

# Get labels + their counts
type_vc = ros_balanced_df['label'].value_counts(sort=False)
total = type_vc.sum()


# Pie chart
fig = px.pie(type_vc, 
    values=type_vc.values, 
    names=type_vc.index, 
    hole=0.500
    )
fig.update_layout(
    title='Data Distribution of Main Dataset', 
    font_size=10, 
    title_x=0.40,
    annotations=[dict(
        text='Main Dataset', 
        font_size=12, 
        showarrow=False, 
        height=5000, 
        width=5000)]
    )
fig.update_traces(
    textfont_size=8, 
    textinfo='label+percent+text',
    text=[f"1:{round(total / v)}" for v in type_vc.values] # Get ratio for every value
    ) 
fig.show()

## 4. Image Overview

### Sizes

In [ ]:
# Get width + height
def get_dims(file_path):
    path = Path(file_path).as_posix()
    img = Image.open(path)
    width, height = img.size
    img.close()
    return width, height


# Holds width, height + label
data = {}


# Add to data variable
for path, label in new_main_df.values:

    # Get the width + height
    width, height = get_dims(path)

    # Check if trash type exists
    if label not in data:
        data[label] = {'heights': [], 'widths': []}

    # Add the width + height
    data[label]['heights'].append(height)
    data[label]['widths'].append(width)


# Plot class image sizes
for label, dims in data.items():

    # Convert to numpy arrays
    w, h = np.array(dims['widths']), np.array(dims['heights'])

    # Normalize in one line 
    # norm_w = (w - w.min()) / np.where(w == 0, 1, w)
    # norm_h = (h - h.min()) / np.where(h == 0, 1, h)

    # Scatter plot image sizes
    plt.figure(figsize=(6, 4))
    plt.scatter(w, h, alpha=0.5)
    plt.xlim(0, w.max())
    plt.ylim(0, h.max())
    plt.title(f'Image Sizes for: {label}')
    plt.xlabel('Width')
    plt.ylabel('Height')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()


print("Complete!")

# raise('ahhh')

print('test')

### Random Sample

In [ ]:
# Random Sample from each class
number_of_samples = 1


# Get random path per label
sample_df = ros_balanced_df.groupby('label').sample(n=number_of_samples).reset_index()


# Get labels + paths
labels = sample_df['label'].tolist()
paths = sample_df['path'].tolist()


# Setup our image grid
num_labels = len(labels) # Number of labels
cols = 2 # Images per row
rows = (num_labels // cols) + (num_labels % cols > 0) # Number of total rows


# Make plots to insert images
fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=labels
)


# Add our chosen images
for i, path in enumerate(paths):

    # For each image
    pth = Path(path).as_posix() # Get path
    with Image.open(path) as img:

        # Convert to RGB
        img_array = np.array(img.convert("RGB"))

    # Choose row + Column for image
    row = (i // cols) + 1
    col = (i % cols) + 1

    # Add image at (row, col)
    fig.add_trace(
        go.Image(z=img_array),
        row=row,
        col=col
    )


# Clean up layout
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(
    height=300 * rows, # Scaling height by number of rows
    width=800,
    title_text="Random Sample from Each Trash Category",
    showlegend=False
)
fig.show()

---
# Step 3: Apply Techniques
Fixing our data

## Technique 1: Geometric Transformations
Fix various problems with the image, such as:
- Size problems
- Rotation problems
- Skewed images
- etc.

### Resizing Images
going through each folder and resizing the image

In [ ]:
# Size of images
width = 256
height = 256


# Resize all images
for folder in os.listdir(main_data_location):

    # For every image, Resize
    print(f"Resized {folder}")
    for image in os.listdir(f"{main_data_location}/{folder}"):
        image_path = os.path.join(main_data_location, folder, image) # Path to the image
        image = Image.open(image_path).convert("RGB") # Load the image in RGB
        image = image.resize( (width, height) ) # Resize using high‑quality downsampling

        # Save the image
        image.save(image_path, format="JPEG")
        image.close()

### Check our Images again
Similar to Step 3: Check our images again

In [ ]:
# Holds width, height + label
data = {}


# Add to data variable
for path, label in ros_balanced_df.values:

    # Get the width + height
    width, height = get_dims(path)

    # Check if trash type exists
    if label not in data:
        data[label] = {'heights': [], 'widths': []}

    # Add the width + height
    data[label]['heights'].append(height)
    data[label]['widths'].append(width)


# Plot class image sizes
for label, dims in data.items():

    # Convert to numpy arrays
    w, h = np.array(dims['widths']), np.array(dims['heights'])

    # Normalize in one line 
    # norm_w = (w - w.min()) / np.where(w == 0, 1, w)
    # norm_h = (h - h.min()) / np.where(h == 0, 1, h)

    # Scatter plot image sizes
    plt.figure(figsize=(6, 4))
    plt.scatter(w, h, alpha=0.5)
    plt.xlim(0, w.max())
    plt.ylim(0, h.max())
    plt.title(f'Image Sizes for: {label}')
    plt.xlabel('Width')
    plt.ylabel('Height')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

print("Complete!")

## Technique 2: Image Segmentation
- Manually Separating object from background using thresholding and Morphological Operations
- Can be replaced by AutoEncoders

In [ ]:

# Define new folder
threshold_data_location = f"{created_dir}/ThresholdMainData"


# Loop through the class folder
if not os.path.exists(threshold_data_location):
    # Segments Image Objects and Backgrounds
    for folder in os.listdir(path=main_data_location):

        # Create the same class folders
        threshold_folder_path = f"{threshold_data_location}/{folder}"
        os.makedirs(threshold_folder_path, exist_ok=True)

        # For every image
        for image in os.listdir(path=f"{main_data_location}/{folder}"):

            # Get the paths
            path = f"{main_data_location}/{folder}/{image}"
            save_path = f"{threshold_folder_path}/{image}"






            # # Setting up variables
            # img = cv2.imread(path) # Image in Blue, Green, Red
            # gry = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) # Gray Image
            # hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV) # Hue, Saturation, Value Image
            # h, s, v = cv2.split(hsv_img) # Split up channels

            # # Gaussian Blur
            # blurred = cv2.GaussianBlur(gry, (5, 5), 0)

            # # Edge Detection
            # edges = cv2.Canny(blurred, 50, 150)


            # # Morphological Operations:

            # # Make edges thicker
            # kernel = np.ones((3,3), np.uint8)
            # edges_dilated = cv2.dilate(edges, kernel, iterations=1)

            # # Find the outlines using edges
            # contours, _ = cv2.findContours(edges_dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # # Fill outlines with white
            # binary_mask = np.zeros_like(gry)
            # cv2.drawContours(binary_mask, contours, -1, (255), thickness=cv2.FILLED)



            # # Joining background and colored object
            # colored_part = cv2.bitwise_and(img, img, mask=binary_mask)
            # bg_mask = cv2.bitwise_not(binary_mask)
            # background_gray = cv2.bitwise_and(gry, gry, mask=bg_mask)
            # background_3_channel = cv2.cvtColor(background_gray, cv2.COLOR_GRAY2BGR)

            # # Final result
            # result = cv2.add(colored_part, background_3_channel)

            # # Save to image
            # cv2.imwrite(save_path, result)
            
            
            
            
            
            

            image = io.imread(path) # Load the array into 'img'

            # 2. TEAMMATE's THRESHOLDING LOGIC
            gray = rgb2gray(image)
            
            blurred_gray = gaussian(gray, sigma=1)
            
            # Local adaptive threshold (Sauvola)
            sauvola_t = threshold_sauvola(blurred_gray, window_size=27, k=0.3) 
            final_mask = blurred_gray < sauvola_t

            # 3. APPLY MASK (Zeroes out the background to black)
            masked_rgb = image.copy()
            masked_rgb[~final_mask] = 0 

            # 4. SAVE RESULT
            # img_as_ubyte converts float arrays back to standard 8-bit image formats safely
            io.imsave(save_path, img_as_ubyte(masked_rgb))







    print("Thresholding complete")


# Path exists already
else:
    print("ThresholdMainData already exists")


image_paths, labels = [], [] # Clear our arrays

# Make df from folders + images
for folder in os.listdir(threshold_data_location): # For folders in RealWaste dataset

    # For every image
    for image in os.listdir(threshold_data_location + "/" + folder):

        # Get image paths
        image_paths.append(os.path.join(threshold_data_location + "/" + folder, image))

        # Add it to a list
        labels.append(folder)

# Make dataframe from paths + labels
threshold_df = pd.DataFrame({"path": image_paths, "label": labels})

# threshold_dfl = DFDataset(threshold_df, transform=aug)


print(threshold_df.head())

In [ ]:
# Separate paths and labels safely
X = threshold_df[['path']]
y = threshold_df['label']

# Apply imblearn's fit resample
X_ros, y_ros = ros.fit_resample(X, y)

# Combine back into a dataframe
threshold_ros_df = X_ros.copy()
threshold_ros_df['label'] = y_ros

# Show new balanced class counts
print(threshold_ros_df['label'].value_counts())

# Print total resampled image count
print(f"Total threshold ROS training images: {len(threshold_ros_df)}")

## Technique 3: Histogram of Oriented Gradients
- Originally used with SVMs and Random Forest
- Made obsolete by CNNs
- Represents images as 1D array, which is then fed into the model
- Works well with CNNs

In [ ]:
# Random Sample from each class
number_of_samples = 1


# Get random path per label
sample_df = ros_balanced_df.groupby('label').sample(n=number_of_samples).reset_index()


# Get labels + paths
labels = sample_df['label'].tolist()
paths = sample_df['path'].tolist()


# Setup our image grid
num_labels = len(labels) # Number of labels
cols = 2 # Images per row
rows = (num_labels // cols) + (num_labels % cols > 0) # Number of total rows


# Make plots to insert images
fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=labels
)


# Histogram of Oriented Gradients (HOG)
for i, path in enumerate(paths):
    
    # Get our image (height, width, 3)
    img = cv2.imread(path) 

    # Get the HOG for image
    _, hog_image = hog(
        img,
        orientations=8,
        pixels_per_cell=(16, 16),
        cells_per_block=(1, 1),
        visualize=True,
        channel_axis=-1,
    )

    # Make subplots with 2 columns
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)

    # Convert to Gray
    ax1.axis('off')
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Plot original image
    ax1.imshow(rgb, cmap='gray')
    ax1.set_title('Input image')

    # Rescale histogram for better display
    hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

    # Plot HOG + show
    ax2.axis('off')
    ax2.imshow(hog_image_rescaled, cmap=plt.cm.gray)
    ax2.set_title('Histogram of Oriented Gradients')
    plt.show()

## Technique 4: Feature Extraction (Segmentation AutoEncoder)
Used for better model performance
- Separates object from background
- Usually better than traditional methods like Image Segmentation
- Enhances HOG

In [ ]:
# Initialize U2 model safely
gpu_session = new_session("u2net", providers=['CUDAExecutionProvider'])

# Start main data background removal
main_out = f"{created_dir}/AutoEncoderMainData"

# Create main output directory
os.makedirs(main_out, exist_ok=True)

# Gather all main image paths
main_tasks = []

# Loop through original class folders
for folder in os.listdir(main_data_location):

    # Read files in the folder
    for img in os.listdir(os.path.join(main_data_location, folder)):

        # Store folder and image names
        main_tasks.append((folder, img))

# Process main images with tqdm
if not os.path.exists(main_out):
    for folder, image in tqdm(main_tasks, desc="Processing Main Data"):

        # Create main class folders safely
        main_class_path = os.path.join(main_out, folder)
        os.makedirs(main_class_path, exist_ok=True)

        # Get exact original image paths
        orig_path = os.path.join(main_data_location, folder, image)

        # Read original image safely
        img = cv2.imread(orig_path)

        # Skip if image is missing
        if img is None: continue

        # Remove background from original image
        clean_img = remove(img, session=gpu_session)

        # Force output to PNG format
        base_name, _ = os.path.splitext(image)
        save_path = os.path.join(main_class_path, f"{base_name}.png")

        # Save clean image to disk
        cv2.imwrite(save_path, clean_img)

else:
    print(f"{main_out} exists already!")

main_paths, main_labels = [], []

# Make dataframe from main folders
for folder in os.listdir(main_out):

    # Loop over every main image
    for image in os.listdir(os.path.join(main_out, folder)):

        # Get clean image full paths
        main_paths.append(os.path.join(main_out, folder, image))

        # Add label to our list
        main_labels.append(folder)

# Make dataframe from main paths
autoencoder_df = pd.DataFrame({"path": main_paths, "label": main_labels})

print(f"Total main images: {len(autoencoder_df)}")

### Fix imbalances with autoencoder_df

In [ ]:
# Separate paths and labels safely
X = autoencoder_df[['path']]
y = autoencoder_df['label']

# Apply imblearn's fit resample
X_ros, y_ros = ros.fit_resample(X, y)

# Combine back into a dataframe
autoencoder_ros_df = X_ros.copy()
autoencoder_ros_df['label'] = y_ros

# Show new balanced class counts
print(autoencoder_ros_df['label'].value_counts())

# Print total resampled image count
print(f"Total ROS training images: {len(autoencoder_ros_df)}")

---
# Step 4: Image specific EDA
Exploring our data further

## 1. Understanding Color in our Images

### Color Histogram

In [ ]:
# Holds the channels of all Images
total_hist_b = np.zeros((256, 1))
total_hist_g = np.zeros((256, 1))
total_hist_r = np.zeros((256, 1))


# Adding images to images list
for folder in os.listdir(main_data_location):



    # For every image, add to histogram
    for image in os.listdir(f"{main_data_location}/{folder}"):
        
        # Get image path
        path = f"{main_data_location}/{folder}/{image}"

        # Get image
        img = cv2.imread(path)

        # Calculate histogram for image
        hist_b = cv2.calcHist([img], [0], None, [256], [0, 256])
        hist_g = cv2.calcHist([img], [1], None, [256], [0, 256])
        hist_r = cv2.calcHist([img], [2], None, [256], [0, 256])

        # Add to channel holders
        total_hist_b += hist_b
        total_hist_g += hist_g
        total_hist_r += hist_r


# Plot blue channel
plt.plot(total_hist_b, color='b')
plt.title('Image Histogram For Blue Channel')
plt.show()


# Plot green channel
plt.plot(total_hist_g, color='g')
plt.title('Image Histogram For Green Channel')
plt.show()


# Plot red channel
plt.plot(total_hist_r, color='r')
plt.title('Image Histogram For Red Channel')
plt.show()


# Plot all channels together
plt.plot(total_hist_b, color='b', label='Blue')
plt.plot(total_hist_g, color='g', label='Green')
plt.plot(total_hist_r, color='r', label='Red')
plt.title('Histogram of all Color Channels')
plt.legend()
plt.show()

### Color Histogram (Alternative)

In [ ]:
# Holds the channels of all Images
total_hist_b = np.zeros((256, 1))
total_hist_g = np.zeros((256, 1))
total_hist_r = np.zeros((256, 1))


# Adding images to images list
for folder in os.listdir(main_data_location):

    # For every image, add to histogram
    for image in os.listdir(f"{main_data_location}/{folder}"):
        
        # Get image path
        path = f"{main_data_location}/{folder}/{image}"

        # Get image
        img = cv2.imread(path)

        # Calculate histogram for image
        hist_b = cv2.calcHist([img], [0], None, [256], [0, 256])
        hist_g = cv2.calcHist([img], [1], None, [256], [0, 256])
        hist_r = cv2.calcHist([img], [2], None, [256], [0, 256])

        # Add to channel holders
        total_hist_b += hist_b
        total_hist_g += hist_g
        total_hist_r += hist_r


# Plot blue channel
plt.figure()
plt.subplot(3, 1, 1)
plt.title("Image Histogram For Blue Channel")
plt.hist(total_hist_b, color="blue")

# Plot green channel
plt.figure()
plt.subplot(3, 1, 1)
plt.title("Image Histogram For Green Channel")
plt.hist(total_hist_g, color="green")

# Plot red channel
plt.figure()
plt.subplot(3, 1, 1)
plt.title("Image Histogram For Red Channel")
plt.hist(total_hist_r, color="red")

# For a clearer view
plt.tight_layout()
plt.show()

# Plot all channels together
plt.title("Histogram of all RGB Colors")
plt.hist(total_hist_b, color="blue")
plt.hist(total_hist_g, color="green")
plt.hist(total_hist_r, color="red")
plt.show()

### Color Histograms for every class

In [ ]:
# Adding images to images list
for folder in os.listdir(main_data_location):

    # Holds the channels of all Images
    total_hist_b = np.zeros((256, 1))
    total_hist_g = np.zeros((256, 1))
    total_hist_r = np.zeros((256, 1))

    # For every image, add to histogram
    for image in os.listdir(f"{main_data_location}/{folder}"):
        
        # Get image path
        path = f"{main_data_location}/{folder}/{image}"

        # Get image
        img = cv2.imread(path)

        # Calculate histogram for image
        hist_b = cv2.calcHist([img], [0], None, [256], [0, 256])
        hist_g = cv2.calcHist([img], [1], None, [256], [0, 256])
        hist_r = cv2.calcHist([img], [2], None, [256], [0, 256])

        # Add to channel holders
        total_hist_b += hist_b
        total_hist_g += hist_g
        total_hist_r += hist_r

    # Plot all channels together
    plt.plot(total_hist_b, color='b', label='Blue')
    plt.plot(total_hist_g, color='g', label='Green')
    plt.plot(total_hist_r, color='r', label='Red')
    plt.title(f'Histogram of {folder} Color Channels')
    plt.legend()
    plt.show()

### Color Histograms (Alternative) for every class

In [ ]:
# Adding images to images list
for folder in os.listdir(main_data_location):

    # Holds the channels of all Images
    total_hist_b = np.zeros((256, 1))
    total_hist_g = np.zeros((256, 1))
    total_hist_r = np.zeros((256, 1))

    # For every image, add to histogram
    for image in os.listdir(f"{main_data_location}/{folder}"):
        
        # Get image path
        path = f"{main_data_location}/{folder}/{image}"

        # Get image
        img = cv2.imread(path)

        # Calculate histogram for image
        hist_b = cv2.calcHist([img], [0], None, [256], [0, 256])
        hist_g = cv2.calcHist([img], [1], None, [256], [0, 256])
        hist_r = cv2.calcHist([img], [2], None, [256], [0, 256])

        # Add to channel holders
        total_hist_b += hist_b
        total_hist_g += hist_g
        total_hist_r += hist_r

    # Plot all channels together
    plt.title(f"Histogram of {folder} RGB Colors")
    plt.hist(total_hist_b, color="blue")
    plt.hist(total_hist_g, color="green")
    plt.hist(total_hist_r, color="red")
    plt.show()

### Brightness Outliers

In [ ]:
# Get average pixel brightness
ros_balanced_df['brightness'] = ros_balanced_df['path'].apply(lambda x: cv2.imread(x, cv2.IMREAD_GRAYSCALE).mean()) # Get mean from grayscale

# Define z-score outliers
z_threshold = 3
global_mean = ros_balanced_df['brightness'].mean()
global_std = ros_balanced_df['brightness'].std()
global_lower = global_mean - z_threshold * global_std
global_upper = global_mean + z_threshold * global_std

z_scores = (ros_balanced_df['brightness'] - global_mean) / global_std
outliers = ros_balanced_df[(z_scores.abs() > z_threshold)]

# Create a histogram
plt.figure(figsize=(15, 6))
plt.hist(ros_balanced_df['brightness'], bins=50, color='skyblue', edgecolor='black')

# Highlight the outliers (using dynamic global values)
plt.axvspan(0, global_lower, color='red', alpha=0.3, label=f'Dark Outliers < {round(global_lower, 2)}')
plt.axvspan(global_upper, 255, color='orange', alpha=0.3, label=f'Bright Outliers > {round(global_upper, 2)}')

# Add plot labels
plt.title('Image Brightness Distribution (All labels)')
plt.xlabel('Average Pixel Brightness')
plt.ylabel('Number of Images')
plt.legend()
plt.show()

# Loop through unique labels for Histograms
for folder in ros_balanced_df['label'].unique():
    folder_data = ros_balanced_df[ros_balanced_df['label'] == folder]['brightness']
    f_mean = folder_data.mean()
    f_std = folder_data.std()
    f_lower = f_mean - z_threshold * f_std
    f_upper = f_mean + z_threshold * f_std

    # Create a histogram
    plt.figure(figsize=(15, 6))
    plt.hist(folder_data, bins=50, color='skyblue', edgecolor='black')

    # Highlight the outliers (fixed labels to be folder-specific)
    plt.axvspan(0, f_lower, color='red', alpha=0.3, label=f'Dark Outliers < {round(f_lower, 2)}')
    plt.axvspan(f_upper, 255, color='orange', alpha=0.3, label=f'Bright Outliers > {round(f_upper, 2)}')

    # Add plot labels
    plt.title(f'Image Brightness Distribution for {folder}')
    plt.xlabel('Average Pixel Brightness')
    plt.ylabel('Number of Images')
    plt.legend()
    plt.show()

# Create a Scatter plot
plt.figure(figsize=(15, 6))

# Separate normal data from outliers (replaced 240 with global_upper)
normal_data = ros_balanced_df[(ros_balanced_df['brightness'] >= global_lower) & (ros_balanced_df['brightness'] <= global_upper)]

# Plot the normal images in blue
plt.scatter(range(len(normal_data)), normal_data['brightness'],
            color='blue', alpha=0.3, s=10, label='Normal Images')

# Plot the outliers in red (replaced 240 with global_upper)
plt.scatter(range(len(normal_data), len(normal_data) + len(outliers)),
            outliers['brightness'],
            color='red', s=25, label='Outliers')

# Draw threshold lines
plt.axhline(y=global_lower, color='black', linestyle='--', alpha=0.5)
plt.axhline(y=global_upper, color='black', linestyle='--', alpha=0.5)

# Add plot labels
plt.title('Scatterplot of Brightness (All labels)')
plt.xlabel('Image Index')
plt.ylabel('Average Pixel Brightness')
plt.legend()
plt.show()

# Loop through unique labels for Scatter plots
for folder in ros_balanced_df['label'].unique():
    folder_mask = ros_balanced_df['label'] == folder
    f_mean = ros_balanced_df[folder_mask]['brightness'].mean()
    f_std = ros_balanced_df[folder_mask]['brightness'].std()
    f_lower = f_mean - z_threshold * f_std
    f_upper = f_mean + z_threshold * f_std

    # Create a Scatter plot
    plt.figure(figsize=(15, 6))

    # Separate normal data from outliers (replaced 240 with f_upper and fixed logic)
    normal_data = ros_balanced_df[folder_mask & (ros_balanced_df['brightness'] >= f_lower) & (ros_balanced_df['brightness'] <= f_upper)]
    outliers = ros_balanced_df[folder_mask & ((ros_balanced_df['brightness'] < f_lower) | (ros_balanced_df['brightness'] > f_upper))]

    # Plot the normal images in blue
    plt.scatter(range(len(normal_data)), normal_data['brightness'],
            color='blue', alpha=0.3, s=10, label='Normal Images')

    # Plot the outliers in red
    plt.scatter(range(len(normal_data), len(normal_data) + len(outliers)),
            outliers['brightness'],
            color='red', s=25, label='Outliers')

    # Draw threshold lines (fixed to use folder-specific stats)
    plt.axhline(y=f_upper, color='black', linestyle='--', alpha=0.5)
    plt.axhline(y=f_lower, color='black', linestyle='--', alpha=0.5)

    # Add plot labels
    plt.title(f'Scatterplot of Brightness for {folder}')
    plt.xlabel('Image Index')
    plt.ylabel('Average Pixel Brightness')
    plt.legend()
    plt.show()

## 2. Understanding Texture (GLCM)
- Color and shape are not enough...
- Enter GLCM (Gray-Level Co-occurrence Matrix): this helps the model analyze the roughness or smoothness of the material

In [ ]:
# Define GLCM feature extractor
def get_glcm_features(image_path):
    # Read image in Grayscale
    img_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    # Return zeros if none
    if img_gray is None:
        return np.zeros(4, dtype=np.float32)
        
    # Resize to standardize texture
    img_gray = cv2.resize(img_gray, (224, 224))
    
    # Compute the GLCM matrix 
    glcm = graycomatrix(img_gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    
    # Extract texture properties
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    
    # Return flat numpy array
    return np.array([contrast, correlation, energy, homogeneity], dtype=np.float32)

In [ ]:
# Find quartiles and IQR bounds
def print_boxplot_stats(data, feature_name):
    # Calculate the 25th percentile
    q1 = np.percentile(data, 25)
    
    # Calculate the 75th percentile
    q3 = np.percentile(data, 75)
    
    # Find the interquartile range
    iqr = q3 - q1
    
    # Calculate the lower boundary
    lower_bound = q1 - (1.5 * iqr)
    
    # Calculate the upper boundary
    upper_bound = q3 + (1.5 * iqr)
    
    # Find the outlier data points
    outliers = [x for x in data if x < lower_bound or x > upper_bound]
    
    # Print out the calculated stats
    print(f"  {feature_name}:")
    print(f"    Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f}")
    print(f"    Outlier Bounds: {lower_bound:.2f} to {upper_bound:.2f}")
    print(f"    Total Outliers Detected: {len(outliers)}\n")

# Hold all class features
all_contrasts = []
all_homogeneities = []
all_energies = []
all_correlations = []

# Iterate through every folder
for folder in os.listdir(main_data_location):
    # Hold single class features
    contrasts = []
    homogeneities = []
    energies = []
    correlations = []

    # Loop through folder images
    for image in os.listdir(f"{main_data_location}/{folder}"):
        # Get full image path
        path = f"{main_data_location}/{folder}/{image}"

        # Extract features using function
        features = get_glcm_features(path)

        # Skip if image missing
        if np.all(features == 0):
            continue

        # Unpack the returned features
        con, cor, ene, hom = features

        # Add to individual lists
        contrasts.append(con)
        correlations.append(cor)
        energies.append(ene)
        homogeneities.append(hom)

        # Add to all classes
        all_contrasts.append(con)
        all_correlations.append(cor)
        all_energies.append(ene)
        all_homogeneities.append(hom)

    # Print math before plotting
    print(f"=== STATS FOR {folder.upper()} ===")
    print_boxplot_stats(contrasts, "Contrast")
    print_boxplot_stats(homogeneities, "Homogeneity")
    print_boxplot_stats(energies, "Energy")
    print_boxplot_stats(correlations, "Correlation")
    print("==================================\n")

    # Plot features in separate subplots
    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    fig.suptitle(f"Texture Features of {folder}", fontsize=14, fontweight="bold")

    # Plot the contrast features
    axes[0].boxplot(contrasts)
    axes[0].set_title("Contrast")
    axes[0].set_xticks([])  

    # Plot the homogeneity features
    axes[1].boxplot(homogeneities)
    axes[1].set_title("Homogeneity")
    axes[1].set_xticks([])

    # Plot the energy features
    axes[2].boxplot(energies)
    axes[2].set_title("Energy")
    axes[2].set_xticks([])

    # Plot the correlation features
    axes[3].boxplot(correlations)
    axes[3].set_title("Correlation")
    axes[3].set_xticks([])

    # Adjust layout and show
    plt.tight_layout()
    plt.show()

# Plot all classes together
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Texture Features of All Classes", fontsize=14, fontweight="bold")

# Plot combined contrast features
axes[0].boxplot(all_contrasts)
axes[0].set_title("Contrast")
axes[0].set_xticks([])

# Plot combined homogeneity features
axes[1].boxplot(all_homogeneities)
axes[1].set_title("Homogeneity")
axes[1].set_xticks([])

# Plot combined energy features
axes[2].boxplot(all_energies)
axes[2].set_title("Energy")
axes[2].set_xticks([])

# Plot combined correlation features
axes[3].boxplot(all_correlations)
axes[3].set_title("Correlation")
axes[3].set_xticks([])

# Adjust layout and show
plt.tight_layout()
plt.show()

## 3. Frequency Domain Analysis (Using Fast Fourier Transform)
Understanding the image through the frequency Spectrum

In [ ]:
# Define the three test resolutions
resolutions = {
    "Extremely Low": (32, 32), 
    "Regular": (224, 224), 
    "Extremely High": (1024, 1024)
}

# Iterate through every folder
for folder in os.listdir(main_data_location):
    
    # Get the folder path
    folder_path = f"{main_data_location}/{folder}"
    
    # Get all image names
    all_images = os.listdir(folder_path)
    
    # Skip empty folders safely
    if not all_images:
        continue
        
    # Pick one random image
    random_image = random.choice(all_images)
    
    # Get full image path
    path = f"{folder_path}/{random_image}"

    # Read image in grayscale
    img_raw = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    # Skip if image missing
    if img_raw is None:
        continue

    # Create a 2x3 plot grid
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Set the main title
    fig.suptitle(f"Resolution Comparison: {folder.upper()}", fontsize=16, fontweight="bold")

    # Loop through the three resolutions
    for i, (res_name, size) in enumerate(resolutions.items()):
        
        # Resize to specific dimensions
        img_resized = cv2.resize(img_raw, size)

        # Apply Fast Fourier Transform
        f_transform = np.fft.fft2(img_resized)

        # Shift the low frequencies
        f_shift = np.fft.fftshift(f_transform)

        # Calculate the magnitude spectrum
        magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1)

        # Plot the resized image
        axes[0, i].imshow(img_resized, cmap='gray')
        axes[0, i].set_title(f"{res_name} Image\n({size[0]}x{size[1]})")
        axes[0, i].axis('off')

        # Plot the magnitude spectrum
        axes[1, i].imshow(magnitude_spectrum, cmap='gray')
        axes[1, i].set_title(f"{res_name} Spectrum")
        axes[1, i].axis('off')

    # Adjust layout and show
    plt.tight_layout()
    plt.show()

## 3. Observing Random Outliers

In [ ]:
# Grab 5 dark and 5 light
dark_outliers = ros_balanced_df[ros_balanced_df['brightness'] < 20].head(5)
light_outliers = ros_balanced_df[ros_balanced_df['brightness'] > 240].head(5)

# Check how many we found
n_dark = len(dark_outliers)
n_light = len(light_outliers)

# Plot our 10 images
if n_dark == 0 and n_light == 0: # If no outliers...
    print("No outliers found! Your dataset has no extreme dark or light images based on these thresholds.")

# Else, plot our 10 images
else:

    # 2 row, 5 column grid
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle("5 Darkest vs 5 Brightest Images", fontsize=16)

    # Plot Dark Outliers
    for i in range(5):
        
        # Plot on first row
        ax = axes[0, i] 

        # Read and Plot image
        if i < n_dark:
            row = dark_outliers.iloc[i]
            img = cv2.imread(row['path'])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Fix colors for Matplotlib
            ax.imshow(img)
            ax.set_title(f"Dark ({row['brightness']:.1f} brightness (intensity))\n{row['label']}")
        ax.axis('off') # Hide grid lines and numbers

    # 4. Plot Light Outliers (Bottom Row / Index 1)
    for i in range(5):

        # Plot on second row
        ax = axes[1, i]

        # Read and Plot image
        if i < n_light:
            row = light_outliers.iloc[i]
            img = cv2.imread(row['path'])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Fix colors for Matplotlib
            ax.imshow(img)
            ax.set_title(f"Light ({row['brightness']:.1f})\n{row['label']}")
        ax.axis('off') # Hide grid lines and numbers

    plt.tight_layout()
    plt.show()


# Drop brightness column
ros_balanced_df = ros_balanced_df.drop(columns="brightness")


# Info on ros_balanced_df
print(ros_balanced_df.info())

## 4. Using Fiftyone

In [ ]:
# Delete dataset if it exists
dataset_name = "Balanced_Dataset"
if fo.dataset_exists(dataset_name):
    fo.delete_dataset(dataset_name)


# Make a new Fiftyone dataset
dataset = fo.Dataset(name=dataset_name)


# Build samples using df
samples = []
for path, label in ros_balanced_df.values:
    
    clean_filepath = str(path)

    # # Load the image path
    sample = fo.Sample(filepath=clean_filepath)

    # Check if failed to recognize image
    if sample.media_type == "unknown":
        # Print the exact string
        print(f"UNKNOWN PATH ENCOUNTERED: '{clean_filepath}' (Length: {len(clean_filepath)})")
        continue  # Skip adding this sample

    # Attach your trash class!
    sample["ground_truth"] = fo.Classification(label=label)
    
    # Add to samples
    samples.append(sample)


# Add all the images
dataset.add_samples(samples)


# Start fiftyone in notebook
session = fo.launch_app(dataset, port=8080)

---
# Step 5: Training

Creating, Testing and Saving our model

## 1. Training

In [ ]:
# Container for our data
datablock = DataBlock(
    blocks=(
        ImageBlock,
        CategoryBlock,
    ),
    get_x=get_x,
    get_y=get_y,
    item_tfms=RandomResizedCrop(224, min_scale=0.8),
    batch_tfms=aug
)


# Wrapper with our data
ros_dls = datablock.dataloaders(
    ros_balanced_df,
    bs=48,
    num_workers=8,  # start here, adjust up/down
    pin_memory=True,
    persistent_workers=True,
)

# do the same for other techniques...
smote_dls = datablock.dataloaders(
    smote_balanced_df,
    bs=48,
    num_workers=8,  # start here, adjust up/down
    pin_memory=True,
    persistent_workers=True,
)

deep_smote_dls = datablock.dataloaders(
    deep_smote_balanced_df,
    bs=48,
    num_workers=8,  # start here, adjust up/down
    pin_memory=True,
    persistent_workers=True,
)

adasyn_dls = datablock.dataloaders(
    adasyn_balanced_df,
    bs=48,
    num_workers=8,  # start here, adjust up/down
    pin_memory=True,
    persistent_workers=True,
)

weighted_rs_dls = datablock.dataloaders(
    train_df,
    bs=48,
    num_workers=8,  # start here, adjust up/down
    pin_memory=True,
    persistent_workers=True,
    sampler=sampler # sampler comes from step 3.2
)

threshold_ros_dls = datablock.dataloaders(
    threshold_df,
    bs=48,
    num_workers=8,  # start here, adjust up/down
    pin_memory=True,
    persistent_workers=True,
)

autoencoded_ros_dls = datablock.dataloaders(
    autoencoder_df,
    bs=48,
    num_workers=8,  # start here, adjust up/down
    pin_memory=True,
    persistent_workers=True,
)


# Test Dataloader
ros_test_dl = ros_dls.test_dl(test_df, with_labels=True)

# Validation Dataloader
ros_valid_dl = ros_dls.test_dl(valid_df, with_labels=True)


# do the same for other techniques...
smote_test_dl = smote_dls.test_dl(test_df, with_labels=True)
smote_valid_dl = smote_dls.test_dl(valid_df, with_labels=True)

deep_smote_test_dl = deep_smote_dls.test_dl(test_df, with_labels=True)
deep_smote_valid_dl = deep_smote_dls.test_dl(valid_df, with_labels=True)

adasyn_test_dl = adasyn_dls.test_dl(test_df, with_labels=True)
adasyn_valid_dl = adasyn_dls.test_dl(valid_df, with_labels=True)

weighted_test_dl = weighted_rs_dls.test_dl(test_df, with_labels=True)
weighted_valid_dl = weighted_rs_dls.test_dl(valid_df, with_labels=True)

threshold_test_dl = threshold_ros_dls.test_dl(test_df, with_labels=True)
threshold_valid_dl = threshold_ros_dls.test_dl(valid_df, with_labels=True)

autoencoded_test_dl = autoencoded_ros_dls.test_dl(test_df, with_labels=True)
autoencoded_valid_dl = autoencoded_ros_dls.test_dl(valid_df, with_labels=True)

In [ ]:
ros_dls.show_batch(nrows=3, ncols=3) # lets show some data
smote_dls.show_batch(nrows=3, ncols=3)
deep_smote_dls.show_batch(nrows=3, ncols=3)
adasyn_dls.show_batch(nrows=3, ncols=3)
weighted_rs_dls.show_batch(nrows=3, ncols=3)
threshold_ros_dls.show_batch(nrows=3, ncols=3)
autoencoded_ros_dls.show_batch(nrows=3, ncols=3)

In [ ]:
images_to_show = 3

# See augmentations
weighted_rs_dls.train.show_batch(max_n=images_to_show)

In [ ]:
# Define techniques and architectures
techniques = ["ROS", "SMOTE", "ADASYN", "Weighted_RS", "Autoencoded_Data", "Sauvola", "DeepSMOTE"]
models = ["EfficientNet", "ResNet-50", "MobileNet"]

# Loop over each technique
for tech in techniques:

    # Make directory for technique
    tech_dir = f"models/{tech}"
    os.makedirs(tech_dir, exist_ok=True)

    # Loop over each model
    for mod in models:

        # Define final file path
        model_path = f"{tech_dir}/{mod}.pkl"

        # Skip if already exists
        if os.path.isfile(model_path):
            print(f"{tech} - {mod}: model is trained!")
            continue

        print(f"Training {mod} on {tech}...")

        # Match string to dataloader
        match tech:
            case "ROS":
                current_dls = ros_dls
            case "SMOTE":
                current_dls = smote_dls
            case "DeepSMOTE":
                current_dls = deep_smote_dls
            case "ADASYN":
                current_dls = adasyn_dls
            case "Weighted_RS":
                current_dls = weighted_rs_dls
            case "Autoencoded_Data":
                current_dls = autoencoded_ros_dls
            case "Sauvola":
                current_dls = threshold_ros_dls

        # Match string to architecture
        match mod:
            case "ResNet-50":
                arch = resnet50
            case "EfficientNet":
                arch = 'efficientnet_b0'
            case "MobileNet":
                arch = 'mobilenetv3_large_100'

        # Initialize the vision learner
        learner = vision_learner(current_dls, arch, metrics=accuracy)

        # Remove standard progress bar
        learner.remove_cb(ProgressCallback)

        # Fine tune our model
        learner.fine_tune(3, base_lr=3e-3, freeze_epochs=1)

        # Export the final model
        learner.export(model_path)

## 2. Quick Predictions

### Quick single image test

In [ ]:
model_path = f"models/ROS/ResNet-50.pkl"
single_learner = load_learner(model_path)
single_learner.dls = ros_dls

# Testing our model on adidas
myimage = f"{main_data_location}/0/0_0.jpg"


# Predict image waste type
img = PILImage.create(myimage) # Create image
pred_class, _, probabilities = single_learner.predict(img) # feed in image

# Get probabilities
probs_dict = {label: probability.item() * 100 for label, probability in zip(single_learner.dls.vocab, probabilities)} # For every label, store its probability
sorted_probs = sorted(probs_dict.items(), key=lambda x: x[1], reverse=True) # Sort the probabilities


# Output probabilities and prediction
plt.imshow(img)
print(f"Prediction: {pred_class}\n")
print("--- List of predictions ---")
for class_name, prob in sorted_probs:
    print(f"{class_name:<20} {prob:>6.2f}%")

### Quick Multi image test

In [ ]:
# Get final metrics on the blind test set
metrics = single_learner.validate(dl=ros_valid_dl)

print(f"Final Test Metrics:\n\nTest Loss: {(metrics[0])}\nAccuracy: {(metrics[1])}")

### Evaluation

In [ ]:
single_learner.show_results(ds_idx=1)

In [ ]:
# Plot the models worst performances (ros resnet-50)
interp = ClassificationInterpretation.from_learner(single_learner)
interp.plot_top_losses(3, nrows=3)
interp.print_classification_report()

---
# Step 6: Analysis
Observing how our model predicts

---
## 1. Shapley Analysis
What is our model looking at?

In [ ]:
# Define technique and model lists
techniques = ["ROS", "SMOTE", "ADASYN", "Weighted_RS", "Autoencoded_Data", "Sauvola", "DeepSMOTE"]
models = ["EfficientNet", "ResNet-50", "MobileNet"]

# Create folder for SHAP plots
os.makedirs("shap_plots", exist_ok=True)

# Set specific class to test
class_to_test = '1'

# Loop over every data technique
for tech in techniques:

    # Loop over every model architecture
    for mod in models:

        # Define specific model file path
        model_path = f"models/{tech}/{mod}.pkl"

        # Check if model file exists
        if os.path.isfile(model_path):

            # Print current model being evaluated
            print(f"Generating SHAP for {tech} {mod}...")

            # Load the saved model file
            learner = load_learner(model_path)

            # Filter dataframe for specific class
            class_df = valid_df[valid_df['label'] == class_to_test]

            # Grab first five image paths
            img_paths = class_df['path'].values[:5]

            # Convert images to numpy arrays
            X = np.stack([
                cv2.resize(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB), (256, 256)) 
                for p in img_paths
            ])

            # Define tensor conversion function
            def f(x):
                # Convert to float tensor safely
                x_tensor = torch.tensor(x).float()
                # Permute the tensor dimensions safely
                x_tensor = x_tensor.permute(0, 3, 1, 2)
                # Scale pixel values down safely
                x_tensor = x_tensor / 255.0
                # Move tensor to GPU safely
                x_tensor = x_tensor.to(learner.dls.device)
                # Return numpy array of predictions
                return learner.model(x_tensor).detach().cpu().numpy()

            # Build the image masker safely
            masker = shap.maskers.Image("blur(128,128)", X[0].shape)

            # Get unique labels from dataframe
            # unique_labels = test_df['label'].unique()
            
            unique_labels = list(learner.dls.vocab)

            # Initialize the SHAP explainer safely
            explainer = shap.Explainer(f, masker, output_names=unique_labels)

            # Calculate the SHAP values safely
            shap_values = explainer(
                X[0:4],
                max_evals=500,
                batch_size=50,
                outputs=shap.Explanation.argsort.flip[0:4],
            )

            # Generate the SHAP image plot
            shap.image_plot(shap_values, show=False)

            # Save plot to hard drive
            plt.savefig(f"shap_plots/{tech}_{mod}.png", bbox_inches='tight')

            # Clear matplotlib figure safely
            plt.clf()

---
## 2. Cam-Grad Visualization
Similar to Shapley Analysis (how does our model predict)
- produces heatmaps to visualize which image parts most influenced a CNN's decision

In [ ]:
# Define techniques and model lists
techniques = ["ROS", "SMOTE", "ADASYN", "Weighted_RS", "Autoencoded_Data", "Sauvola", "DeepSMOTE"]
models = ["EfficientNet", "ResNet-50", "MobileNet"]

# Create folder for Grad-CAM plots
os.makedirs("gradcam_plots", exist_ok=True)

# Define the Grad-CAM class safely
class SimpleGradCAM:
    def __init__(self, model, layer):
        # Store the model reference safely
        self.model = model
        # Hook the forward pass safely
        layer.register_forward_hook(self.save_act)
        # Hook the backward pass safely
        layer.register_full_backward_hook(self.save_grad)

    # Save the output activations safely
    def save_act(self, m, i, o): self.act = o

    # Save the output gradients safely
    def save_grad(self, m, i, o): self.grad = o[0]

    # Calculate the class activation map
    def get_cam(self, x, cls_idx):
        # Zero out the model gradients
        self.model.zero_grad()
        # Run the forward pass safely
        out = self.model(x)
        # Run the backward pass safely
        out[0, cls_idx].backward()
        # Average the feature gradients safely
        w = self.grad.mean(dim=(2, 3), keepdim=True)
        # Weight the output activations safely
        cam = (w * self.act).sum(1).squeeze()
        # Detach and move to CPU
        cam = cam.detach().cpu().numpy()
        # Keep positive values only safely
        return np.maximum(cam, 0)

# Loop over every data technique
for tech in techniques:
    # Loop over every model architecture
    for mod in models:
        # Define specific model file path
        model_path = f"models/{tech}/{mod}.pkl"

        # Check if model file exists
        if os.path.isfile(model_path):
            # Print current model being evaluated
            print(f"Generating Grad-CAM for {tech} {mod}...")

            # Load the saved model file
            learner = load_learner(model_path)

            # Get all dataset classes safely
            classes = list(learner.dls.vocab)

            # Set up model device safely
            dev = learner.dls.device
            learner.model.to(dev)

            try:
                # Target last CNN layer safely
                target = learner.model[0][-1]
                # Initialize the Grad-CAM object
                gcam = SimpleGradCAM(learner.model, target)
            except:
                # Skip if architecture is incompatible
                print(f"Skipping {mod}: layer structure unsupported.")
                continue

            # Set up plot size safely
            plt.figure(figsize=(10, 4 * len(classes)))

            # Loop over all dataset classes
            for i, cls_name in enumerate(classes):
                # Filter dataframe for specific class
                class_df = valid_df[valid_df['label'] == cls_name]

                # Get the first image path
                path = class_df['path'].iloc[0]

                # Load the image with OpenCV
                img = cv2.imread(str(path))
                # Convert color format to RGB
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # Prepare image for the model
                x = torch.tensor(img).float()
                # Permute the tensor dimensions safely
                x = x.permute(2, 0, 1)
                # Scale between zero and one
                x = x.div(255.0).unsqueeze(0).to(dev)

                # Generate the raw heatmap safely
                cam = gcam.get_cam(x, i)

                # Resize to original dimensions safely
                cam = cv2.resize(cam, (img.shape[1], img.shape[0]))
                # Scale heatmap zero to one
                cam = cam / (cam.max() + 1e-10)

                # Create heatmap color overlay safely
                hm = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
                # Convert color format to RGB
                hm = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
                # Combine image and heatmap safely
                overlay = cv2.addWeighted(img, 0.6, hm, 0.4, 0)

                # Plot the original image safely
                plt.subplot(len(classes), 2, 2*i + 1)
                plt.imshow(img); plt.axis('off')
                plt.title(f"Original: {cls_name}")

                # Plot the heatmap overlay safely
                plt.subplot(len(classes), 2, 2*i + 2)
                plt.imshow(overlay); plt.axis('off')
                plt.title(f"Grad-CAM: {cls_name}")

            # Adjust the plot layout safely
            plt.tight_layout()
            # Save plot to hard drive
            plt.savefig(f"gradcam_plots/{tech}_{mod}.png", bbox_inches='tight')
            # Clear matplotlib figure safely
            plt.clf()
            # Close the figure completely safely
            plt.close()

# Step 7: Final Test

---
## 1. Validation results
from valid_dl

In [ ]:
# Define techniques and models
techniques = ["ROS", "SMOTE", "ADASYN", "Weighted_RS", "Autoencoded_Data", "Sauvola", "DeepSMOTE"]
models = ["EfficientNet", "ResNet-50", "MobileNet"]

# Initialize empty results list
results_list = []

# Iterate over data techniques
for tech in techniques:

    # Match dataloader to technique
    match tech:
        case "ROS":
            valid_dl = ros_valid_dl
        case "SMOTE":
            valid_dl = smote_valid_dl
        case "DeepSMOTE":
            valid_dl = smote_valid_dl
        case "ADASYN":
            valid_dl = adasyn_valid_dl
        case "Weighted_RS":
            valid_dl = weighted_valid_dl
        case "Autoencoded_Data":
            valid_dl = autoencoded_valid_dl
        case "Sauvola":
            valid_dl = threshold_valid_dl

    # Iterate through model architectures
    for mod in models:

        # Define exact model path
        model_path = f"models/{tech}/{mod}.pkl"

        # Check if file exists
        if os.path.isfile(model_path):

            # Load saved learner file
            learner = load_learner(model_path)

            # Get predictions and targets
            preds, targs = learner.get_preds(dl=valid_dl)

            # Convert probabilities to classes
            preds = preds.argmax(dim=1)

            # Generate dictionary metrics report
            report = classification_report(targs, preds, output_dict=True, zero_division=0)

            # Extract weighted average metrics
            avg_prec = report['weighted avg']['precision']
            avg_rec = report['weighted avg']['recall']
            avg_f1 = report['weighted avg']['f1-score']
            avg_supp = report['weighted avg']['support']

            # Validate on test set
            metrics = learner.validate(dl=valid_dl)

            # Extract loss and accuracy
            test_loss = metrics[0]
            test_acc = metrics[1]

            # Append metrics to results
            results_list.append({
                "Technique": tech,
                "Architecture": mod,
                "Test Loss": round(test_loss, 4),
                "Test Accuracy (%)": round(test_acc * 100, 2),
                "Precision": round(avg_prec, 4),
                "Recall": round(avg_rec, 4),
                "F1 Score": round(avg_f1, 4),
                "Support": int(avg_supp)
            })

        else:
            # Append missing model placeholders
            results_list.append({
                "Technique": tech,
                "Architecture": mod,
                "Test Loss": None,
                "Test Accuracy (%)": None,
                "Precision": None,
                "Recall": None,
                "F1 Score": None,
                "Support": None
            })

# Convert results to dataframe
results_df = pd.DataFrame(results_list)

In [ ]:
results_df.sort_values(by=['Test Accuracy (%)'], ascending=False)

In [ ]:
worse_oversampling_techniques = ["ADASYN", "SMOTE", "ROS"]
no_oversample_df = results_df[~results_df['Technique'].isin(worse_oversampling_techniques)]
no_oversample_df.sort_values(by=['Test Accuracy (%)'], ascending=False)

---
## 2. Test results
from test_dl

In [ ]:
# Define techniques and models
techniques = ["ROS", "SMOTE", "ADASYN", "Weighted_RS", "Autoencoded_Data", "Sauvola", "DeepSMOTE"]
models = ["EfficientNet", "ResNet-50", "MobileNet"]

# Initialize empty results list
results_list = []

# Iterate over data techniques
for tech in techniques:

    # Match dataloader to technique
    match tech:
        case "ROS":
            test_dl = ros_test_dl
        case "SMOTE":
            test_dl = smote_test_dl
        case "DeepSMOTE":
            test_dl = smote_test_dl
        case "ADASYN":
            test_dl = adasyn_test_dl
        case "Weighted_RS":
            test_dl = weighted_test_dl
        case "Autoencoded_Data":
            test_dl = autoencoded_test_dl
        case "Sauvola":
            test_dl = threshold_test_dl

    # Iterate through model architectures
    for mod in models:

        # Define exact model path
        model_path = f"models/{tech}/{mod}.pkl"

        # Check if file exists
        if os.path.isfile(model_path):

            # Load saved learner file
            learner = load_learner(model_path)

            # Print current analysis header
            print(f"\n{'='*60}")
            print(f"Analyzing: Architecture = {mod} | Technique = {tech}")
            print(f"{'='*60}")

            # Create interpretation from learner
            interp = ClassificationInterpretation.from_learner(learner, dl=test_dl)

            # Plot top three losses
            interp.plot_top_losses(3, nrows=3)

            # Show plot before continuing
            plt.show()

            # Print text classification report
            interp.print_classification_report()

            # Get predictions and targets
            preds, targs = learner.get_preds(dl=test_dl)

            # Convert probabilities to classes
            preds = preds.argmax(dim=1)

            # Generate dictionary metrics report
            report = classification_report(targs, preds, output_dict=True, zero_division=0)

            # Extract weighted average metrics
            avg_prec = report['weighted avg']['precision']
            avg_rec = report['weighted avg']['recall']
            avg_f1 = report['weighted avg']['f1-score']
            avg_supp = report['weighted avg']['support']

            # Validate on test set
            metrics = learner.validate(dl=test_dl)

            # Extract loss and accuracy
            test_loss = metrics[0]
            test_acc = metrics[1]

            # Append metrics to results
            results_list.append({
                "Technique": tech,
                "Architecture": mod,
                "Test Loss": round(test_loss, 4),
                "Test Accuracy (%)": round(test_acc * 100, 2),
                "Precision": round(avg_prec, 4),
                "Recall": round(avg_rec, 4),
                "F1 Score": round(avg_f1, 4),
                "Support": int(avg_supp)
            })

        else:
            # Append missing model placeholders
            results_list.append({
                "Technique": tech,
                "Architecture": mod,
                "Test Loss": None,
                "Test Accuracy (%)": None,
                "Precision": None,
                "Recall": None,
                "F1 Score": None,
                "Support": None
            })

# Convert results to dataframe
results_df = pd.DataFrame(results_list)

# Print final performance table
print(f"\n{'='*60}")
print("FINAL PERFORMANCE METRICS")
print(f"{'='*60}")
print(results_df.to_string(index=False))

In [ ]:
results_df.sort_values(by=['Test Accuracy (%)'], ascending=False)

In [ ]:
worse_oversampling_techniques = ["ADASYN", "SMOTE", "ROS"]
no_oversample_df = results_df[~results_df['Technique'].isin(worse_oversampling_techniques)]
no_oversample_df.sort_values(by=['Test Accuracy (%)'], ascending=False)